# How does the per-segment detrend order change the low-frequency spectrum of a signal with a quadratic trend?

A slow polynomial trend is not stationary noise. Within each segment it appears as an offset, a ramp, and a curvature, and after windowing their power falls into the lowest bins, where it can exceed the noise by orders of magnitude. Removing a polynomial from each segment before the transform suppresses that leakage, provided the polynomial order matches what the trend looks like within a segment. In a logarithmically scheduled spectrum the segments are longest at the lowest frequencies, which is where an under-fitted trend leaves the largest residue. The question is which detrend order brings the low-frequency spectrum down to the noise.

For orders 1 and 2, SpecKit projects each segment onto an orthonormal polynomial basis $Q$ from `_build_Q`, built on $t \in [-1, 1]$, and removes that projection:

$$y_{\rm detr} = y - Q\,Q^{\mathsf T} y.$$

The signal is sampled at $f_s = 1024$ Hz for 16 s. It contains sinusoids of amplitude 0.2 at 7.5 Hz and 0.1 at 55 Hz, white noise of standard deviation 0.003, and the trend $0.8 - 0.5\,t + 0.01\,t^2$ with $t$ in seconds. `compute_spectrum(x, fs, order=o)` runs for `o` in `-1, 0, 1, 2`: `-1` applies the window alone, `0` removes the segment mean, and `1` and `2` remove a linear or quadratic polynomial. The other settings are the defaults: a Kaiser window at 200 dB peak sidelobe level, its default overlap, and the `vectorized_ltf` scheduler. The spectra are one-sided ASDs. The orders are compared with each other and with the level of the injected white noise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from speckit import compute_spectrum, default_rc

plt.rcParams.update(default_rc)

In [ ]:
# Generate synthetic data with quadratic trend
np.random.seed(7)
fs = 1024.0  # Hz
T = 16.0  # seconds
N = int(T * fs)
t = np.arange(N) / fs

# A couple of tones + white noise
noise = 0.003 * np.random.randn(N)
sig = 0.2 * np.sin(2 * np.pi * 7.5 * t) + 0.1 * np.sin(2 * np.pi * 55.0 * t)

# Quadratic trend a0 + a1*t + a2*t^2
a0, a1, a2 = 0.8, -0.5, 0.01
trend = a0 + a1 * t + a2 * (t**2)

x = sig + noise + trend

# Compute spectra for orders -1, 0, 1, 2
orders = [-1, 0, 1, 2]
results = {}
for o in orders:
    res = compute_spectrum(x, fs, order=o)
    results[o] = (res.f, res.asd)

# %% [4] Plot ASD overlays
plt.figure(figsize=(7.5, 5))
for o in orders:
    f, A = results[o]
    if o == 2:
        ls = "--"
    else:
        ls = "-"
    plt.loglog(f, A, label=f"order={o}", lw=2, ls=ls)
plt.xlabel("Frequency [Hz]")
plt.ylabel("ASD [units/√Hz]")
plt.title("Effect of polynomial detrending on ASD")
plt.grid(True, which="both")
plt.legend()
plt.tight_layout()
plt.show()

## Reading the result

The single figure overlays the ASD for the four orders, with order 2 dashed. At the two tones and at high frequency the four traces should coincide, because the trend does not reach those bins. At the lowest frequencies they should separate in order: order −1 carries the offset, ramp, and curvature; order 0 removes the offset; order 1 removes the ramp; order 2 removes the whole trend, because a global quadratic is quadratic within every segment.

The comparison that decides the notebook is the order-2 trace against the injected noise floor. White noise of standard deviation $\sigma$ at $f_s$ has the one-sided ASD $\sigma\sqrt{2/f_s} = 0.003\sqrt{2/1024} \approx 1.3\times10^{-4}$ units/$\sqrt{\text{Hz}}$. Order 2 should reach that level down to the lowest bins, and the lower orders should rise above it there.

Limits in the code: the record is 16 s, so the lowest bin is near $f_s/N = 1/16$ Hz. The tone peaks are ASD values in bins whose width changes with frequency, so their heights are not the tone amplitudes.